# 04 - Modeling: Developer careers and compensation
**Modeling by:** Mansoor Mohiuddin

**Question:** How do experience, job role, and work arrangement relate to reported yearly compensation, and can simple models predict it in dollars?

**Data:** Stack Overflow Developer Survey 2025 (one row per respondent). Respondents are self-selected and many skip the pay question, so results describe *this sample*, not all developers, and are associations, not guaranteed salary effects.

**Input:** `data/processed/us_fulltime_clean.csv`, the cleaned file written by notebook 01 (4,435 rows). This notebook does not re-clean the raw survey, so it uses the same respondents as the EDA notebooks.

In [ ]:
# ====== EDIT THESE ======
DATA_FILE = "data/processed/us_fulltime_clean.csv"   # cleaned file written by notebook 01 (already in the repo)
DATA_URL = "https://raw.githubusercontent.com/samia21-04/Developer_Compensation/main/" + DATA_FILE
RANDOM_STATE = 42
# ========================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 1. Load the prepared data and choose the predictors
The cleaning is done once, in notebook 01, so every notebook uses the same respondents. Choices (write these in the report):
- **Population:** 4,435 employed respondents in the United States who reported compensation, after notebook 01 kept pay from $10,000 up to the 99th percentile of reporters ($641,900).
- **Target:** `pay` = `ConvertedCompYearly`, yearly total compensation already converted to USD by Stack Overflow, so one consistent currency.
- **Left out of the predictors:** `log_pay` (the same amount in another form), `ResponseId` (an ID), and `exp_band` / `RemoteWork_detail` (regroupings of predictors we already use).
- **Experience:** `WorkExp_num`, years of professional work experience. It is already a number in 2025; the older `YearsCodePro` text column no longer exists.
- **Role:** in 2025 each respondent picks one `DevType`. Notebook 01 grouped the answers into 9 role families (`role`).
- **Missing values:** a few predictors are blank (35 experience, 17 work arrangement, 5 education, 4 organization size). They are filled inside the pipeline in section 3, using training data only.

In [ ]:
# Use the repo copy if it is here (run from notebooks/ or the repo root); otherwise read the same file from GitHub.
local_copies = [p for p in (Path("..") / DATA_FILE, Path(DATA_FILE)) if p.exists()]
source = local_copies[0] if local_copies else DATA_URL
data = pd.read_csv(source)
print("Loaded:", source)
print("Prepared sample (rows, columns):", data.shape)

TARGET = "pay"   # yearly total compensation in USD
feature_cols = ["WorkExp_num", "role", "RemoteWork", "EdLevel", "OrgSize"]
missing = [c for c in feature_cols + [TARGET] if c not in data.columns]
assert not missing, f"Columns not found: {missing}. Re-run notebook 01 to rebuild the processed file."

df = data[feature_cols + [TARGET]].copy()
print("Final modeling table:", df.shape)
print("Missing values per column:")
print(df.isna().sum())

In [ ]:
# Quick look for the report: medians AND sample sizes (not just means)
band_order = ["0-2", "3-5", "6-10", "11-20", "20+"]   # "20+" means more than 20 years
print(data.groupby("exp_band")[TARGET].agg(n="count", median="median", mean="mean").round(0).reindex(band_order))
print()
print(data.groupby("RemoteWork")[TARGET].agg(n="count", median="median", mean="mean").round(0))

## 2. Split into train / test
The test set is locked away until the end. Models never learn from it.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,}")

## 3. Preprocessing pipeline
Same cleaning every time: fill missing values, scale numbers, turn categories into 0/1 columns.
The pipeline is fit on training data only, so nothing leaks from the test set.

In [ ]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_steps = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_steps = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_steps, num_cols),
    ("cat", categorical_steps, cat_cols),
])
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

## 4. Scoring helper (results in dollars)

In [ ]:
results = []

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results.append({"Model": name, "MAE ($)": mae, "RMSE ($)": rmse})
    return model, pred

## 5. Baseline: always guess the median training pay
Every real model has to beat this.

In [ ]:
baseline, _ = evaluate("Median baseline", DummyRegressor(strategy="median"))
results[-1]

## 6. Model 1: Linear regression

In [ ]:
lin, lin_pred = evaluate("Linear regression", Pipeline([
    ("prep", preprocess),
    ("model", LinearRegression()),
]))
results[-1]

## 7. Model 2: KNN regression
We pick the number of neighbors (k) using cross-validation on the **training** data only.

In [ ]:
knn_search = GridSearchCV(
    Pipeline([("prep", preprocess), ("model", KNeighborsRegressor())]),
    param_grid={"model__n_neighbors": [3, 5, 10, 15, 25]},
    scoring="neg_mean_absolute_error", cv=5,
)
knn, knn_pred = evaluate("KNN regression", knn_search)
print("Best k:", knn_search.best_params_["model__n_neighbors"])
results[-1]

## 8. Model 3: Regression tree
We limit the tree's depth (using cross-validation) so it doesn't just memorize the training data.

In [ ]:
tree_search = GridSearchCV(
    Pipeline([("prep", preprocess), ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))]),
    param_grid={"model__max_depth": [3, 4, 5, 6, 8, 10],
                "model__min_samples_leaf": [5, 10, 20]},
    scoring="neg_mean_absolute_error", cv=5,
)
tree, tree_pred = evaluate("Regression tree", tree_search)
print("Best settings:", tree_search.best_params_)
results[-1]

## 9. Comparison table (lower is better)

In [ ]:
results_df = (pd.DataFrame(results)
              .sort_values("MAE ($)")
              .reset_index(drop=True))
results_df.style.format({"MAE ($)": "${:,.0f}", "RMSE ($)": "${:,.0f}"})

In [ ]:
FIG_DIR = Path("../figures") if Path("../figures").is_dir() else Path(".")   # repo figures/ folder when run from notebooks/
ax = results_df.set_index("Model")[["MAE ($)", "RMSE ($)"]].plot(kind="bar", figsize=(8, 4))
ax.set_ylabel("Error in dollars")
ax.set_title("Test-set error by model (lower is better)")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FIG_DIR / "model_comparison.png", dpi=200)   # drop this into the report
plt.show()

## 10. Predicted vs. actual (best model)

In [ ]:
best_name = results_df.loc[results_df["Model"] != "Median baseline"].iloc[0]["Model"]
best_pred = {"Linear regression": lin_pred, "KNN regression": knn_pred, "Regression tree": tree_pred}[best_name]

plt.figure(figsize=(5, 5))
plt.scatter(y_test, best_pred, alpha=0.4)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
plt.plot(lims, lims, "r--", label="perfect prediction")
plt.xlabel("Actual ($)"); plt.ylabel("Predicted ($)")
plt.title(f"{best_name}: predicted vs. actual")
plt.legend(); plt.tight_layout()
plt.savefig(FIG_DIR / "best_model_pred_vs_actual.png", dpi=200)
plt.show()

## 11. Plain-language explanations (reuse in the report)

- **Median baseline:** always guess the middle pay from the training data. It ignores every feature, so it is the "no skill" bar to beat.
- **Linear regression:** finds one straight-line recipe, such as "start here, add X dollars per extra unit of feature A", and applies it to every row. Simple, fast, and easy to explain, but it assumes features add up in a straight line.
- **KNN regression:** to predict a new row, find the k most similar respondents in the training data and average their pay. No formula, just "similar respondents report similar pay". Needs scaled features, which our pipeline does.
- **Regression tree:** asks a chain of yes/no questions (for example "is experience more than 10 years?") and lands on a group of similar rows, then predicts that group's average pay. Can capture non-straight-line patterns and is easy to read, but can overfit if allowed to grow too deep.

**Why these fit the problem:** we are predicting a number (yearly pay in dollars), so we need regression models. These three go from simplest to most flexible, so we can see whether extra complexity actually helps.

**How to read the results:** MAE is the typical miss in dollars. RMSE punishes large misses more. A model is only useful if both are clearly below the baseline.